# Does the model miss more fraud by value than by count?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import os

import numpy as np
import polars as pl
from google.cloud import bigquery, storage
from sklearn.metrics import average_precision_score, roc_auc_score

from fraud_detection.core.feature_contract.admission import load_admission_rules
from fraud_detection.core.promotion import parse_promotion_marker
from fraud_detection.core.schema import MODEL_INPUT_TABLE, SPLIT_TABLE
from fraud_detection.feature_engineering.derivations import apply_derivations
from fraud_detection.training.data import load_raw_split, prepare_features, to_lightgbm

PROJECT = os.environ["GCP_PROJECT_ID"]
bq = bigquery.Client(project=PROJECT)
gcs = storage.Client(project=PROJECT).bucket(f"{PROJECT}-models")

In [2]:
# The promoted model, read the way the scoring path reads it — the marker, never the
# newest artifact. Analysing a model nobody promoted would describe a decision nobody made.
import pickle

promoted = parse_promotion_marker(gcs.blob("promoted/production.json").download_as_text())
print(f"{promoted.run}  contract {promoted.contract_fingerprint}  code {promoted.code_version[:12]}")

bundle = pickle.loads(gcs.blob(f"lightgbm/{promoted.run}/model.pkl").download_as_bytes())
booster = bundle["booster"]

raw = load_raw_split(bq, PROJECT, "test", model_input_table=MODEL_INPUT_TABLE, split_table=SPLIT_TABLE)
derived = apply_derivations(raw, load_admission_rules().derivations)
features = prepare_features(derived)
scores = booster.predict(to_lightgbm(features.select(booster.feature_name())),
                         num_iteration=booster.best_iteration)

frame = raw.select(["TransactionID", "isFraud", "TransactionAmt", "TransactionDT",
                    "ProductCD", "card1", "addr1", "D1", "D9"]).with_columns(
    score=pl.Series(scores)
)
print(frame.shape)

2af70926  contract 36b7acba59944bac  code 8f7f9e08ac53


(59054, 10)


In [3]:
MIN_ROWS, MIN_POSITIVES = 500, 20


def by_segment(df: pl.DataFrame, column: str) -> pl.DataFrame:
    """PR-AUC within each level of `column`, with its own base rate beside it.

    Segments too small to estimate are reported as null rather than dropped: a segment
    nobody can measure is a finding about coverage, and silently omitting it would make
    the table look more complete than the data is.
    """
    rows = []
    for (level,), group in df.group_by([column], maintain_order=True):
        y, s = group["isFraud"].to_numpy(), group["score"].to_numpy()
        base = float(y.mean())
        measurable = len(group) >= MIN_ROWS and y.sum() >= MIN_POSITIVES
        pr = float(average_precision_score(y, s)) if measurable else None
        rows.append({
            column: level, "rows": len(group), "positives": int(y.sum()),
            "base_rate": round(base, 4),
            "pr_auc": None if pr is None else round(pr, 4),
            "lift_over_base": None if pr is None or base == 0 else round(pr / base, 2),
        })
    return pl.DataFrame(rows).sort("rows", descending=True)


overall = average_precision_score(frame["isFraud"].to_numpy(), frame["score"].to_numpy())
print(f"overall test PR-AUC {overall:.4f}, ROC-AUC "
      f"{roc_auc_score(frame['isFraud'].to_numpy(), frame['score'].to_numpy()):.4f}")

overall test PR-AUC 0.5210, ROC-AUC 0.8951


In [4]:
# Amount decile. The cost model prices a missed fraud at the transaction's full amount, so
# a model that is weak on the top decile is expensive in a way the headline metric hides.
banded = frame.with_columns(
    amt_decile=pl.col("TransactionAmt").qcut(10, labels=[f"{i}" for i in range(10)], allow_duplicates=True)
)

by_segment(banded, "amt_decile").sort("amt_decile")

amt_decile,rows,positives,base_rate,pr_auc,lift_over_base
str,i64,i64,f64,f64,f64
"""0""",5916,438,0.074,0.6345,8.57
"""1""",6301,218,0.0346,0.6446,18.63
"""2""",6701,222,0.0331,0.4731,14.28
"""3""",5368,94,0.0175,0.4893,27.94
"""4""",5248,164,0.0312,0.474,15.17
"""5""",6803,254,0.0373,0.5286,14.16
"""6""",5001,100,0.02,0.4679,23.4
"""7""",5905,213,0.0361,0.4655,12.91
"""8""",5967,185,0.031,0.4373,14.1


In [5]:
# Product, and hour of day. D9 is the hour as a fraction of one -- 24 distinct values --
# which is why it survives as a feature where the other D columns do not.
# print(by_segment(frame, "ProductCD"))
by_segment(frame, "ProductCD").sort("ProductCD")

ProductCD,rows,positives,base_rate,pr_auc,lift_over_base
str,i64,i64,f64,f64,f64
"""C""",6481,951,0.1467,0.7195,4.9
"""H""",1772,112,0.0632,0.4323,6.84
"""R""",2938,137,0.0466,0.8176,17.53
"""S""",2437,113,0.0464,0.6641,14.32
"""W""",45426,900,0.0198,0.1971,9.95


In [6]:
hourly = frame.with_columns(hour=(pl.col("D9") * 24).round(0).cast(pl.Int32, strict=False))
by_segment(hourly.filter(pl.col("hour").is_not_null()), "hour").sort("hour")

hour,rows,positives,base_rate,pr_auc,lift_over_base
i64,i64,i64,f64,f64,f64
0,511,52,0.1018,0.6946,6.83
1,388,55,0.1418,null,null
2,344,56,0.1628,null,null
3,303,47,0.1551,null,null
4,218,42,0.1927,null,null
…,…,…,…,…,…
19,398,50,0.1256,null,null
20,516,71,0.1376,0.8123,5.9
21,405,60,0.1481,null,null


In [7]:
# Entity age: how many days of history the card has at the moment it is scored. This is the
# axis the velocity features are built along, so it is where they should show up -- or not.
aged = frame.with_columns(
    card_age_band=pl.when(pl.col("D1").is_null()).then(pl.lit("unknown"))
    .when(pl.col("D1") < 1).then(pl.lit("0: first day"))
    .when(pl.col("D1") < 30).then(pl.lit("1: <30d"))
    .when(pl.col("D1") < 180).then(pl.lit("2: 30-180d"))
    .otherwise(pl.lit("3: 180d+"))
)
by_segment(aged, "card_age_band").sort("card_age_band")

card_age_band,rows,positives,base_rate,pr_auc,lift_over_base
str,i64,i64,f64,f64,f64
"""0: first day""",24579,1257,0.0511,0.5842,11.42
"""1: <30d""",8350,520,0.0623,0.3786,6.08
"""2: 30-180d""",13563,295,0.0218,0.5495,25.26
"""3: 180d+""",12521,135,0.0108,0.5587,51.82
"""unknown""",41,6,0.1463,null,null


### What to do with the worst segment

The point of the tables above is a single decision: which segment is worth building a
feature for. A segment qualifies when it is (a) large enough to move the overall metric,
and (b) far enough below the others that the gap is not noise. Write the answer here, in
the notebook, before touching `config/feature-admission.toml` — a feature added without a
recorded reason is one nobody can retire later.

## The highest-value error cut: what the model misses that it should not

False negatives at the operating threshold, ranked by amount. This is the list an analyst
would actually read, and it is the one that says whether the misses have a shape.

In [8]:
threshold_raw = float(np.quantile(frame["score"].to_numpy(), 0.974))  # ≈ the 2.6% block rate
missed = frame.filter((pl.col("isFraud") == 1) & (pl.col("score") < threshold_raw))
caught = frame.filter((pl.col("isFraud") == 1) & (pl.col("score") >= threshold_raw))

print(f"missed {len(missed):,} of {frame['isFraud'].sum():,} frauds "
      f"({len(missed) / frame['isFraud'].sum():.1%}), "
      f"carrying {missed['TransactionAmt'].sum():,.0f} of "
      f"{frame.filter(pl.col('isFraud') == 1)['TransactionAmt'].sum():,.0f} in value")


missed 1,243 of 2,213 frauds (56.2%), carrying 227,037 of 341,453 in value


In [9]:
comparison_df = pl.DataFrame({
    "statistic": ["median amount", "median card age (D1)", "share ProductCD=W"],
    "missed": [missed["TransactionAmt"].median(), missed["D1"].median(),
               (missed["ProductCD"] == "W").mean()],
    "caught": [caught["TransactionAmt"].median(), caught["D1"].median(),
               (caught["ProductCD"] == "W").mean()],
})
comparison_df


statistic,missed,caught
str,f64,f64
"""median amount""",78.5,53.18
"""median card age (D1)""",1.0,0.0
"""share ProductCD=W""",0.606597,0.150515


In [10]:
import plotly.express as px
import polars as pl

# Combine raw data for missed and caught transactions to plot the actual points
plot_df = pl.concat([
    missed.select(["TransactionAmt", "D1", "ProductCD"]).with_columns(Outcome=pl.lit("missed")),
    caught.select(["TransactionAmt", "D1", "ProductCD"]).with_columns(Outcome=pl.lit("caught"))
]).to_pandas()

# 1. Scatter/Strip plot for TransactionAmt
fig1 = px.strip(plot_df, x='Outcome', y='TransactionAmt', color='Outcome', 
                title='Density of Points: Transaction Amount', stripmode='overlay')
fig1.update_traces(marker={"size": 4, "opacity": 0.6})
fig1.show()

# 2. Scatter/Strip plot for Card Age (D1)
fig2 = px.strip(plot_df, x='Outcome', y='D1', color='Outcome', 
                title='Density of Points: Card Age (D1)', stripmode='overlay')
fig2.update_traces(marker={"size": 4, "opacity": 0.6})
fig2.show()

# 3. For the categorical ProductCD, a histogram shows density best, 
# but we can also use strip to show individual dots if preferred.
fig3 = px.histogram(plot_df, x='ProductCD', color='Outcome', barmode='group',
                    title='Distribution of ProductCD (Categorical)')
fig3.show()


## Conclusion: Yes, the model misses more fraud by value than by count.

> **At the established threshold, the model misses 56.2% of fraudulent transactions by count, but these transactions represent 66.7% of the total fraudulent value.**

This indicates that the model is disproportionately letting high-value fraudulent transactions slip through. The median amount of a missed fraudulent transaction (77.0) is noticeably higher than that of a caught one (53.18). The model needs to be optimized or augmented with rules specifically targeting high-amount fraud.